## 02 · Building Custom Tools

### Why Custom Tools?
Built-in tools are great, but real-world agents need **domain-specific** actions:
- Query your own database
- Call internal APIs
- Run business logic

### 3 Ways to Create Tools

| Method | Best For |
|--------|----------|
| `@tool` decorator | Simple functions, quick setup |
| `@tool` + Pydantic schema | Multiple typed arguments |
| `BaseTool` subclass | Full control, complex logic |

In [1]:
import os
from load_dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("LLM ready")

c:\Users\arunm\OneDrive\Desktop\LangChain_Tutorial\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


LLM ready


## Method 1 — `@tool` Decorator (Simple)
The simplest way. The **docstring** becomes the tool's description — the LLM uses it to decide when to call the tool.

In [2]:
from langchain.tools import tool

@tool
def get_product_price(product_name: str) -> str:
    """Get the current price of a product from our internal catalog.
    Use this tool when the user asks about product prices."""
    catalog = {
        "laptop": "₹75,000",
        "mouse": "₹1,200",
        "keyboard": "₹2,500",
        "monitor": "₹18,000",
        "headphones": "₹3,500",
    }
    key = product_name.lower().strip()
    return catalog.get(key, f"Product '{product_name}' not found in catalog")

# Inspect the tool
print("Tool Name    :", get_product_price.name)
print("Description  :", get_product_price.description)
print("Args Schema  :", get_product_price.args)
print()

# Direct test
print("Direct call  :", get_product_price.invoke("laptop"))

Tool Name    : get_product_price
Description  : Get the current price of a product from our internal catalog.
Use this tool when the user asks about product prices.
Args Schema  : {'product_name': {'title': 'Product Name', 'type': 'string'}}

Direct call  : ₹75,000


## Method 2 — `@tool` with Pydantic Schema
When your tool needs **multiple typed arguments**, use `args_schema` with a Pydantic model for validation.

In [3]:
from pydantic import BaseModel, Field
from langchain.tools import tool

# Define input schema
class OrderInput(BaseModel):
    product_name: str = Field(description="Name of the product to order")
    quantity: int = Field(description="Number of units to order", gt=0)
    customer_id: str = Field(description="Unique customer identifier")

@tool(args_schema=OrderInput)
def place_order(product_name: str, quantity: int, customer_id: str) -> str:
    """Place an order for a product. Use when the user wants to purchase an item."""
    order_id = f"ORD-{customer_id[:3].upper()}-{abs(hash(product_name)) % 9000 + 1000}"
    return (
        f"Order placed successfully!\n"
        f"  Order ID   : {order_id}\n"
        f"  Product    : {product_name} × {quantity}\n"
        f"  Customer   : {customer_id}\n"
        f"  Status     : Processing"
    )

print("Tool Name    :", place_order.name)
print("Args Schema  :", place_order.args)
print()

# Direct test
result = place_order.invoke({"product_name": "Laptop", "quantity": 2, "customer_id": "USR-001"})
print(result)

Tool Name    : place_order
Args Schema  : {'product_name': {'description': 'Name of the product to order', 'title': 'Product Name', 'type': 'string'}, 'quantity': {'description': 'Number of units to order', 'exclusiveMinimum': 0, 'title': 'Quantity', 'type': 'integer'}, 'customer_id': {'description': 'Unique customer identifier', 'title': 'Customer Id', 'type': 'string'}}

Order placed successfully!
  Order ID   : ORD-USR-6916
  Product    : Laptop × 2
  Customer   : USR-001
  Status     : Processing


## Method 3 — `BaseTool` Subclass
Use this for full control: retry logic, async support, logging, custom error handling.

In [4]:
from langchain.tools import BaseTool
from typing import Optional

class InventoryCheckTool(BaseTool):
    name: str = "inventory_check"
    description: str = (
        "Check available stock for a product. "
        "Use when the user asks about stock availability or inventory levels."
    )

    # Simulated warehouse data
    _inventory: dict = {
        "laptop": 15,
        "mouse": 120,
        "keyboard": 45,
        "monitor": 8,
        "headphones": 32,
    }

    def _run(self, product_name: str) -> str:
        """Synchronous execution (required)."""
        key = product_name.lower().strip()
        stock = self._inventory.get(key, None)
        if stock is None:
            return f"Product '{product_name}' not found in inventory"
        status = "In Stock" if stock > 10 else ("Low Stock" if stock > 0 else "Out of Stock")
        return f"{product_name}: {stock} units available — {status}"

inventory_tool = InventoryCheckTool()
print("Tool:", inventory_tool.name)
print()
print(inventory_tool.invoke("laptop"))
print(inventory_tool.invoke("monitor"))
print(inventory_tool.invoke("tablet"))

Tool: inventory_check

laptop: 15 units available — In Stock
monitor: 8 units available — Low Stock
Product 'tablet' not found in inventory


## Combine All Tools → Build an E-Commerce Agent

In [5]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import create_agent

search_tool = DuckDuckGoSearchRun(description="Search the web for product reviews or general info")

ecommerce_toolkit = [
    get_product_price,      # Method 1
    place_order,            # Method 2
    inventory_tool,         # Method 3
    search_tool,            # Built-in
]

agent = create_agent(model=llm, tools=ecommerce_toolkit)

print("E-Commerce Agent ready with tools:")
for t in ecommerce_toolkit:
    print(f"  • {t.name}")

E-Commerce Agent ready with tools:
  • get_product_price
  • place_order
  • inventory_check
  • duckduckgo_search


In [6]:
query = "What is the price and stock status of a laptop? If available, place an order for 1 unit for customer USR-042."

print(f"User: {query}\n")
print("=" * 60)

events = agent.stream(
    {"messages": [("user", query)]},
    stream_mode="values",
)

for event in events:
    event["messages"][-1].pretty_print()

User: What is the price and stock status of a laptop? If available, place an order for 1 unit for customer USR-042.

================================ Human Message =================================

What is the price and stock status of a laptop? If available, place an order for 1 unit for customer USR-042.
================================== Ai Message ==================================
Tool Calls:
  inventory_check (call_8MBSzPqHU37L9Dqhip0deUKh)
 Call ID: call_8MBSzPqHU37L9Dqhip0deUKh
  Args:
    product_name: laptop
================================= Tool Message =================================
Name: inventory_check

laptop: 15 units available — In Stock
================================== Ai Message ==================================
Tool Calls:
  get_product_price (call_D4pVwOVU2IFbynbRJ8gOALqu)
 Call ID: call_D4pVwOVU2IFbynbRJ8gOALqu
  Args:
    product_name: laptop
================================= Tool Message =================================
Name: get_product_price

₹75,000
=

## Tool Best Practices

| Tip | Why |
|-----|-----|
| Write detailed docstrings | LLM reads them to decide when to use the tool |
| Use Pydantic schemas | Prevents wrong arguments being passed by the LLM |
| Return strings | Tools must return strings (or serializable objects) |
| Name tools clearly | `get_product_price` beats `tool1` |
| Handle errors gracefully | Return error as a string, don't raise exceptions |